In [ ]:
%cd ../..
%load_ext autoreload
%autoreload 2

/gpfs/data/schambralab/quantitativeRehabilitation/__lab_member_homes/victor/cvfm4rehab


## Tasks
- Fill out contact in `signal_generator.py`
- Verify prompts are correct (set `include_prompt_info_in_df` in `predict_with_state_machine`) to get the prompts in the resulting dataframe
- Verify videos loaded below
- Run basic contact / idle.

In [27]:
import os
import json
from lmms_eval.tasks.strokerehab.utils_primitives import load_strokerehab_primitives_dataset
from data.utils_strokerehab import DataPaths, PrimitiveLabelUtils

from tools.ultralytics_pose import Pose2DStream
from tools.signal_generator import predict_with_state_machine
import pandas as pd
import matplotlib.pyplot as plt


class DummyVLM:
    def process_frames(self, frames, prompt) -> str:
        print(prompt)
        if "Locate" in prompt:
            return """
[
  {"bbox_2d": [100, 150, 300, 400], "label": "person"},
  {"bbox_2d": [120, 180, 220, 280], "label": "hand"},
  {"bbox_2d": [50, 60, 100, 120], "label": "cup"}
]
"""
        return "IDK"

    def clear(self):
        pass

In [30]:
ds = load_strokerehab_primitives_dataset(
    video_regex=rf'^.*/(C00020|S0001/|S0005|S00021)_.*1_[12]\.(mkv|avi)$'
)
paths = pd.DataFrame(ds['test'])[['path_v', 'path_l']]
path_ls = [os.path.join(DataPaths.RAW_LABEL_DIR, p) for p in paths['path_l'].tolist()]
path_vs = [os.path.join(DataPaths.RAW_VIDEO_DIR, p) for p in paths['path_v'].tolist()]
print("Number of videos:", len(path_vs))
print("First three: ", path_vs[:3])

Number of videos: 52
First three:  ['/gpfs/data/schambralab/quantitativeRehabilitation/__data/VideoData/rawVideosADLsandFM/C00020/C00020_glasses1_1.mkv', '/gpfs/data/schambralab/quantitativeRehabilitation/__data/VideoData/rawVideosADLsandFM/C00020/C00020_RTT right side1_2.avi', '/gpfs/data/schambralab/quantitativeRehabilitation/__data/VideoData/rawVideosADLsandFM/C00020/C00020_drinking1_1.mkv']


In [33]:
path_ls

['/gpfs/data/schambralab/quantitativeRehabilitation/__data/rawVideoLabels/C00020/C00020_glasses1_1.csv',
 '/gpfs/data/schambralab/quantitativeRehabilitation/__data/rawVideoLabels/C00020/C00020_RTT right side1_2.csv',
 '/gpfs/data/schambralab/quantitativeRehabilitation/__data/rawVideoLabels/C00020/C00020_drinking1_1.csv',
 '/gpfs/data/schambralab/quantitativeRehabilitation/__data/rawVideoLabels/C00020/C00020_combing1_2.csv',
 '/gpfs/data/schambralab/quantitativeRehabilitation/__data/rawVideoLabels/C00020/C00020_combing1_1.csv',
 '/gpfs/data/schambralab/quantitativeRehabilitation/__data/rawVideoLabels/C00020/C00020_brushing1_1.csv',
 '/gpfs/data/schambralab/quantitativeRehabilitation/__data/rawVideoLabels/C00020/C00020_RTT left side1_2.csv',
 '/gpfs/data/schambralab/quantitativeRehabilitation/__data/rawVideoLabels/C00020/C00020_face wash1_2.csv',
 '/gpfs/data/schambralab/quantitativeRehabilitation/__data/rawVideoLabels/C00020/C00020_feeding1_1.csv',
 '/gpfs/data/schambralab/quantitativeR

In [3]:
vlm = DummyVLM()
# from tools.vqa.qwen2_5_vl import Qwen2_5_VL_VQA
# vlm = Qwen2_5_VL_VQA(
#     pretrained="Qwen/Qwen2.5-VL-32B-Instruct",
#     device="cuda",
#     device_map=None,
#     use_cache=True,
# )
streamer = Pose2DStream()


In [43]:
path_l, path_v = path_ls[3], path_vs[3]
hand_to_track = PrimitiveLabelUtils.get_handedness(path_l)
df = predict_with_state_machine(
    path_v, path_l, hand_to_track, vlm, streamer, include_prompt_info_in_df=True,
    idle_prompt_methods=("SMC", "Idle", "StatefulIdleFromPred", "StatefulIdleFromGT", "Focus"),
    contact_prompt_methods=(),
    crop_methods=("window", "tracklet")
)

Locate the patient as a bounding box in JSON. If there are multiple people, find all of them.
Processed /gpfs/data/schambralab/quantitativeRehabilitation/__data/VideoData/rawVideosADLsandFM/C00020/C00020_combing1_2.mkv | 0.00-0.27s ...
Processed /gpfs/data/schambralab/quantitativeRehabilitation/__data/VideoData/rawVideosADLsandFM/C00020/C00020_combing1_2.mkv | 0.27-0.53s ...
Processed /gpfs/data/schambralab/quantitativeRehabilitation/__data/VideoData/rawVideosADLsandFM/C00020/C00020_combing1_2.mkv | 0.53-0.80s ...
Processed /gpfs/data/schambralab/quantitativeRehabilitation/__data/VideoData/rawVideosADLsandFM/C00020/C00020_combing1_2.mkv | 0.80-1.07s ...
Processed /gpfs/data/schambralab/quantitativeRehabilitation/__data/VideoData/rawVideosADLsandFM/C00020/C00020_combing1_2.mkv | 1.07-1.33s ...
Processed /gpfs/data/schambralab/quantitativeRehabilitation/__data/VideoData/rawVideosADLsandFM/C00020/C00020_combing1_2.mkv | 1.33-1.60s ...
Processed /gpfs/data/schambralab/quantitativeRehabilit

In [44]:
df.columns

Index(['start_t', 'end_t', 'left_wrist_kps', 'left_elbow_kps', 'left_hand_kps',
       'right_wrist_kps', 'right_elbow_kps', 'right_hand_kps',
       'window_should_infer', 'window_moving_tracklet',
       'window_other_hand_in_view', 'window_bboxes', 'window_SMC',
       'window_Idle', 'window_StatefulIdleFromPred',
       'window_StatefulIdleFromGT', 'window_Focus', 'window_SMC_info',
       'window_Idle_info', 'window_StatefulIdleFromPred_info',
       'window_StatefulIdleFromGT_info', 'window_Focus_info',
       'tracklet_should_infer', 'tracklet_moving_tracklet',
       'tracklet_other_hand_in_view', 'tracklet_bboxes', 'tracklet_SMC',
       'tracklet_Idle', 'tracklet_StatefulIdleFromPred',
       'tracklet_StatefulIdleFromGT', 'tracklet_Focus', 'tracklet_SMC_info',
       'tracklet_Idle_info', 'tracklet_StatefulIdleFromPred_info',
       'tracklet_StatefulIdleFromGT_info', 'tracklet_Focus_info'],
      dtype='object')

In [45]:
df.iloc[0]

start_t                                                                             0.0
end_t                                                                             0.267
left_wrist_kps                        [[1056.0, 265.0, 0.021682914], [1060.0, 271.0,...
left_elbow_kps                        [[1047.0, 238.0, 0.010041503], [1051.0, 248.0,...
left_hand_kps                         [[1062.0, 283.0, 0.010041503], [1066.0, 287.0,...
right_wrist_kps                       [[1036.0, 261.0, 0.67108375], [1035.0, 258.0, ...
right_elbow_kps                       [[1039.0, 233.0, 0.77792233], [1040.0, 233.0, ...
right_hand_kps                        [[1035.0, 280.0, 0.67108375], [1032.0, 275.0, ...
window_should_infer                                                               False
window_moving_tracklet                                                            False
window_other_hand_in_view                                                         False
window_bboxes                   

In [47]:
df.head()

,start_t,end_t,left_wrist_kps,left_elbow_kps,left_hand_kps,right_wrist_kps,right_elbow_kps,right_hand_kps,window_should_infer,window_moving_tracklet,...,tracklet_SMC,tracklet_Idle,tracklet_StatefulIdleFromPred,tracklet_StatefulIdleFromGT,tracklet_Focus,tracklet_SMC_info,tracklet_Idle_info,tracklet_StatefulIdleFromPred_info,tracklet_StatefulIdleFromGT_info,tracklet_Focus_info
0,0.000,0.267,"[[1056.0, 265.0, 0.021682914], [1060.0, 271.0,...","[[1047.0, 238.0, 0.010041503], [1051.0, 248.0,...","[[1062.0, 283.0, 0.010041503], [1066.0, 287.0,...","[[1036.0, 261.0, 0.67108375], [1035.0, 258.0, ...","[[1039.0, 233.0, 0.77792233], [1040.0, 233.0, ...","[[1035.0, 280.0, 0.67108375], [1032.0, 275.0, ...",False,False,...,True,False,False,False,False,"{'prompt': ""Focus on the patient's right hand....","{'prompt': ""Is the patient's right hand idle? ...","{'prompt': ""This clip follows one where the pa...","{'prompt': ""This clip follows one where the pa...","{'prompt': ""Focus ONLY on the patient's right ..."
1,0.267,0.533,"[[nan, nan, 0.0], [1044.0, 234.0, 0.06507788],...","[[nan, nan, 0.0], [1028.0, 264.0, 0.052481923]...","[[nan, nan, 0.0], [1056.0, 213.0, 0.052481923]...","[[nan, nan, 0.0], [1051.0, 234.0, 0.7515595], ...","[[nan, nan, 0.0], [1025.0, 264.0, 0.90932405],...","[[nan, nan, 0.0], [1070.0, 212.0, 0.7515595], ...",False,False,...,True,False,False,False,False,"{'prompt': ""Focus on the patient's right hand....","{'prompt': ""Is the patient's right hand idle? ...","{'prompt': ""This clip follows one where the pa...","{'prompt': ""This clip follows one where the pa...","{'prompt': ""Focus ONLY on the patient's right ..."
2,0.533,0.800,"[[nan, nan, 0.0], [nan, nan, 0.0], [nan, nan, ...","[[nan, nan, 0.0], [nan, nan, 0.0], [nan, nan, ...","[[nan, nan, 0.0], [nan, nan, 0.0], [nan, nan, ...","[[nan, nan, 0.0], [nan, nan, 0.0], [nan, nan, ...","[[nan, nan, 0.0], [nan, nan, 0.0], [nan, nan, ...","[[nan, nan, 0.0], [nan, nan, 0.0], [nan, nan, ...",False,False,...,True,False,False,False,False,"{'prompt': ""Focus on the patient's right hand....","{'prompt': ""Is the patient's right hand idle? ...","{'prompt': ""This clip follows one where the pa...","{'prompt': ""This clip follows one where the pa...","{'prompt': ""Focus ONLY on the patient's right ..."
3,0.800,1.067,"[[nan, nan, 0.0], [1047.0, 254.0, 0.010098983]...","[[nan, nan, 0.0], [1035.0, 251.0, 0.0038429748...","[[nan, nan, 0.0], [1055.0, 256.0, 0.0038429748...","[[nan, nan, 0.0], [1038.0, 249.0, 0.71342206],...","[[nan, nan, 0.0], [1034.0, 247.0, 0.8344833], ...","[[nan, nan, 0.0], [1040.0, 251.0, 0.71342206],...",False,False,...,True,False,False,False,False,"{'prompt': ""Focus on the patient's right hand....","{'prompt': ""Is the patient's right hand idle? ...","{'prompt': ""This clip follows one where the pa...","{'prompt': ""This clip follows one where the pa...","{'prompt': ""Focus ONLY on the patient's right ..."
4,1.067,1.333,"[[nan, nan, 0.0], [1045.0, 252.0, 0.008619394]...","[[nan, nan, 0.0], [1042.0, 246.0, 0.002761127]...","[[nan, nan, 0.0], [1048.0, 256.0, 0.002761127]...","[[nan, nan, 0.0], [1031.0, 249.0, 0.71300995],...","[[nan, nan, 0.0], [1040.0, 241.0, 0.82034016],...","[[nan, nan, 0.0], [1025.0, 255.0, 0.71300995],...",False,False,...,True,False,False,False,False,"{'prompt': ""Focus on the patient's right hand....","{'prompt': ""Is the patient's right hand idle? ...","{'prompt': ""This clip follows one where the pa...","{'prompt': ""This clip follows one where the pa...","{'prompt': ""Focus ONLY on the patient's right ..."
